## IMPORTAR LAS LIBRERIAS

In [1]:
import numpy as np
import pandas as pd
import cloudpickle

#Automcompletar rápido
%config IPCompleter.greedy=True

from janitor import clean_names

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

## CARGAR LOS DATOS

### Ruta del proyecto

In [2]:
ruta_proyecto = 'C:/Users/Lenovo/Documents/portafolio_ds/automation/LEAD_KAGGLE'

### Nombre del archivo de datos

In [3]:
nombre_archivo_datos = 'Lead Scoring.csv'

### Cargar los datos

In [4]:
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_archivo_datos

df = pd.read_csv(ruta_completa,index_col=0)

### Seleccionar solo las variables finales

#### Cargar la lista de variables finales

In [5]:
nombre_variables_finales = ruta_proyecto + '/05_Resultados/' + 'variables_finales.pickle'

pd.read_pickle(nombre_variables_finales).sort_index().index.to_list()

['asymmetrique_activity_score_mms',
 'last_notable_activity_SMS Sent',
 'lead_origin_Lead Add Form',
 'page_views_per_visit_mms',
 'tags_Already a student',
 'tags_Closed by Horizzon',
 'tags_OTHER',
 'tags_Ringing',
 'tags_Will revert after reading the email',
 'total_time_spent_on_website_mms',
 'total_visits_mms']

#### Apuntar (manualmente) la lista de variables finales sin extensiones

In [6]:
variables_finales = ['asymmetrique_activity_score',
                    'last_notable_activity',
                    'lead_origin',
                    'page_views_per_visit',
                    'tags',
                    'total_time_spent_on_website',
                    'total_visits'
                  ]

## ESTRUCTURA DE LOS DATASETS

### Corregir los nombres

In [7]:
df = clean_names(df)

In [8]:
df.rename(columns = {'totalvisits':'total_visits'}, inplace=True)

### Eliminar registros

#### Por duplicados

In [9]:
df.drop_duplicates(inplace = True)

#### Por EDA

In [10]:
#Aunque algunas de estas variables no queden en la selección final, es importante eliminar estos registros porque de lo
#contrario terminaríamos haciendo el perfil de clientes que no quieren ser contactados
df = df.loc[(df.do_not_call !='Yes') & (df.do_not_email !='Yes') & (df.last_activity !='Email Bounced')] 

#### Para x

Quedarse solo con las de la lista.

In [11]:
x = df[variables_finales].copy()

#### Para y

Especificar la target.

In [12]:
target = 'converted'

Crear el y.

In [13]:
y = df[target].copy()

## CREAR EL PIPELINE

### Instanciar calidad de datos

#### Crear la función

In [14]:
def calidad_datos(df):
    temp=df

    #Corrección tipo
    temp = df.astype({'total_visits': 'Int64'})
    
    #Imputar por valor
    var_imputar_valor = ['last_notable_activity',
                         'lead_origin',
                         'tags']
    valor = 'Unknown'
    
    temp[var_imputar_valor] = temp[var_imputar_valor].fillna(valor)
    
    #Imputar por mediana
    var_imputar_mediana =['asymmetrique_activity_score',
                          'page_views_per_visit',
                          'total_time_spent_on_website',
                          'total_visits']
    
    def imputar_mediana(variable):
        if pd.api.types.is_integer_dtype(variable):
            return(variable.fillna(int(variable.median())))
        else:
            return(variable.fillna(variable.median()))
    
    temp[var_imputar_mediana] = temp[var_imputar_mediana].apply(imputar_mediana)
    
    #Imputar raras
    def agrupar_cat_raras(variable, criterio):
        #Calcula las frecuencias
        frecuencias = variable.value_counts(normalize=True)
        #Identifica las que están por debajo del criterio
        temp = [cada for cada in frecuencias.loc[frecuencias < criterio].index.values]
        #Las recodifica en 'OTHER'
        temp2 = np.where(variable.isin(temp),'OTHER',variable)
        #Devuelve el resultado
        return(temp2)

    var_agrupar_cat_raras = ['last_notable_activity',
                             'lead_origin',
                             'tags']
    
    for variable in var_agrupar_cat_raras:
        temp[variable] = agrupar_cat_raras(temp[variable],criterio = 0.02)

    #Winsorización manual
    temp['page_views_per_visit'] = temp['page_views_per_visit'].clip(0, 20)
    temp['total_visits'] = temp['total_visits'].clip(0, 50)
    
    return(temp)

In [15]:
#Comprobando que la función no tenga errores
calidad_datos(x).isna().sum()

asymmetrique_activity_score    0
last_notable_activity          0
lead_origin                    0
page_views_per_visit           0
tags                           0
total_time_spent_on_website    0
total_visits                   0
dtype: int64

#### Convertirla en transformer

In [16]:
hacer_calidad_datos = FunctionTransformer(calidad_datos)

### Instanciar transformación de variables

In [17]:
var_ohe = ['last_notable_activity',
           'lead_origin',
           'tags']

ohe = OneHotEncoder(sparse_output = False, handle_unknown='ignore')


var_mms = ['asymmetrique_activity_score',
           'page_views_per_visit',
           'total_time_spent_on_website',
           'total_visits']

mms = MinMaxScaler()

### Crear el pipe del preprocesamiento

#### Crear el column transformer

In [18]:
ct = make_column_transformer(
    (ohe, var_ohe),
    (mms, var_mms),
    remainder='drop')

#### Crear el pipeline del preprocesamiento

In [19]:
pipe_prepro = make_pipeline(hacer_calidad_datos, 
                            ct)

### Instanciar el modelo

#### Instanciar el algoritmo

In [20]:
modelo = LogisticRegression(n_jobs = -1, 
                       solver = 'saga',
                       penalty = 'l1',
                       C = 1,
                       )

#### Crear el pipe final de entrenamiento

In [21]:
pipe_entrenamiento = make_pipeline(pipe_prepro,modelo)

#### Guardar el pipe final de entrenamiento

In [22]:
nombre_pipe_entrenamiento = 'pipe_entrenamiento.pickle'

ruta_pipe_entrenamiento = ruta_proyecto + '/04_Modelos/' + nombre_pipe_entrenamiento

with open(ruta_pipe_entrenamiento, mode='wb') as file:
   cloudpickle.dump(pipe_entrenamiento, file)

#### Entrenar el pipe final de ejecución

In [23]:
pipe_ejecucion = pipe_entrenamiento.fit(x,y)

## GUARDAR EL PIPE

### Nombre del pipe final de ejecución

In [24]:
nombre_pipe_ejecucion = 'pipe_ejecucion.pickle'

### Guardar el pipe final de ejecución

In [25]:
ruta_pipe_ejecucion = ruta_proyecto + '/04_Modelos/' + nombre_pipe_ejecucion

with open(ruta_pipe_ejecucion, mode='wb') as file:
   cloudpickle.dump(pipe_ejecucion, file)